In [5]:
import pandas as pd
import string, json, re, datetime

def to_float(value):
    """
    Converts a value to float. 
    Returns 0.0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return float(clean_val)
    except (ValueError, TypeError):
        return None

def to_int(value):
    """
    Converts a value to int. 
    Returns 0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return int(clean_val)
    except (ValueError, TypeError):
        return None
    
def to_bool(value):
    """
    Converts a value to boolean.
    Returns None if the value is None or empty.
    """
    if pd.isna(value):
        return None
    
    str_val = str(value).strip()
    
    if str_val.upper() == "N/A":
        return "N/A"
    
    if str_val == "":
        return None
    
    str_val = str(value).lower()
    
    if str_val in ['true', '1', 'yes', 'pass']:
        return True
    elif str_val in ['false', '0', 'no', 'notpass']:
        return False
    else:
        return None

def process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns):
    """
    Flattens all columns in the dataframe into a simple key:value JSON structure.
    """
    df = df.rename(columns=rename_columns)
    exclude_from_data = ["workorder_id", "filename"] + exclude_cols

    for _, row in df.iterrows():
        fname = row.get("filename", "unknown")
        wo_id = str(row.get("workorder_id", None))
        
        if not fname or fname == "nan":
            continue

        if fname not in final_json:
            final_json[fname] = {
                "workorder_id": wo_id,
                "data": {}
            }

        row_flattened_data = {}
        for col in df.columns:
            if col in exclude_from_data:
                continue
            
            value = row[col]
            
            if any(num_col in col for num_col in float_columns):
                row_flattened_data[col] = to_float(value)
            elif isinstance(value, (pd.Timestamp, datetime.datetime, datetime.date)):
                row_flattened_data[col] = value.isoformat()
            elif any(num_col in col for num_col in int_columns):
                row_flattened_data[col] = to_int(value)
            elif any(bool_col in col for bool_col in bool_columns):
                row_flattened_data[col] = to_bool(value)
            else:
                row_flattened_data[col] = value if pd.notna(value) else None
                
        final_json[fname]["data"].update(row_flattened_data)

    return final_json

### WiFi

In [15]:
path = "../../output/snc/wifi.xlsx" 
df = pd.read_excel(path, sheet_name="wifi", keep_default_na=False)

df.drop(columns=["remarks"], inplace=True, errors='ignore')

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
rename_columns = {}
exclude_cols = []

meta_cols = ["workorder_id", "filename", "station", "date_time", "comment_recommendation", "performed_by", "verified_by"]

for col in df.columns:
    if col.endswith("status"):
        bool_columns.append(col)
    
df = df.rename(columns={
    "performed_by": "technician_id",
    "verified_by": "supervisor_id",
})

base_cols = ["workorder_id", "filename", "station", "date_time"]
end_cols = ["comment_recommendation", "technician_id", "supervisor_id"]

ordered_cols = base_cols + bool_columns + end_cols

df = df[ordered_cols]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_wifi.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_wifi.xlsx


### PABX

In [17]:
path = "../../output/snc/pabx.xlsx" 
df = pd.read_excel(path, sheet_name="pabx", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
rename_columns = {}

for col in df.columns:
    new_col = col

    if new_col.startswith("procedures_"):
        new_col = new_col.replace("procedures_", "", 1)

    match = re.match(r"^(.*)_([a-z])$", new_col)
    if match:
        base, suffix = match.groups()
        new_col = f"{base}.{suffix}.status"

    if new_col != col:
        df = df.rename(columns={col: new_col})

df = df.rename(columns={
    "performed_by": "technician_id",
    "verified_by": "supervisor_id",
})

bool_columns = [col for col in df.columns if col.endswith("status")]
    
exclude_cols = [
    "pm_order_no", "reference_document_no"
]

base_cols = ["station", "date_time"]
end_cols = ["comment_recommendation", "technician_id", "supervisor_id"]

maintenance_cols = [
    col for col in df.columns
    if col not in base_cols
    and col not in end_cols
]

ordered_cols = base_cols + maintenance_cols + end_cols

df = df[ordered_cols]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_pabx.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_pabx.xlsx


### OTN

In [18]:
path = "../../output/snc/otn.xlsx" 
df = pd.read_excel(path, sheet_name="otn", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
rename_columns = {}
exclude_cols = []

meta_cols = ["workorder_id", "filename", "station", "date_time", "comment_recommendation", "performed_by", "verified_by"]

for col in df.columns:
    if col not in meta_cols:
        bool_columns.append(col)
    
df = df.rename(columns={
    "performed_by": "technician_id",
    "verified_by": "supervisor_id",
})

base_cols = ["workorder_id", "filename", "station", "date_time"]
end_cols = ["comment_recommendation", "technician_id", "supervisor_id"]

ordered_cols = base_cols + bool_columns + end_cols

df = df[ordered_cols]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_otn.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_otn.xlsx


### PAPIS

In [19]:
path = "../../output/snc/papis.xlsx" 
df = pd.read_excel(path, sheet_name="papis", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
rename_columns = {}
exclude_cols = []

meta_cols = ["workorder_id", "filename", "station", "date_time", "comment_recommendation", "performed_by", "verified_by"]

for col in df.columns:
    if col not in meta_cols:
        bool_columns.append(col)
    
df = df.rename(columns={
    "performed_by": "technician_id",
    "verified_by": "supervisor_id",
})

base_cols = ["workorder_id", "filename", "station", "date_time"]
end_cols = ["comment_recommendation", "technician_id", "supervisor_id"]

ordered_cols = base_cols + bool_columns + end_cols

df = df[ordered_cols]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_papis.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_papis.xlsx


### Projector

In [20]:
path = "../../output/snc/projector.xlsx" 
df = pd.read_excel(path, sheet_name="projector", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
rename_columns = {}
exclude_cols = []

meta_cols = ["workorder_id", "filename", "station", "date_time", "comment_recommendation", "performed_by", "verified_by"]

for col in df.columns:
    if col.endswith("status") and col not in meta_cols:
        bool_columns.append(col)
    if col.endswith("desc") and col not in meta_cols:
        exclude_cols.append(col)
    
df = df.rename(columns={
    "performed_by": "technician_id",
    "verified_by": "supervisor_id",
})

base_cols = ["workorder_id", "filename", "station", "date_time"]
end_cols = ["comment_recommendation", "technician_id", "supervisor_id"]

ordered_cols = base_cols + bool_columns + end_cols

df = df[ordered_cols]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_projector.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_projector.xlsx


### Radio

In [21]:
path = "../../output/snc/radio.xlsx" 
df = pd.read_excel(path, sheet_name="radio", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
rename_columns = {}
exclude_cols = []

meta_cols = ["workorder_id", "filename", "station", "date_time", "comment_recommendation", "performed_by", "verified_by"]

for col in df.columns:
    if col.endswith("status") or col.endswith("remarks"):
        bool_columns.append(col)
    
df = df.rename(columns={
    "performed_by": "technician_id",
    "verified_by": "supervisor_id",
})

base_cols = ["workorder_id", "filename", "station", "date_time"]
end_cols = ["comment_recommendation", "technician_id", "supervisor_id"]

ordered_cols = base_cols + bool_columns + end_cols

df = df[ordered_cols]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_radio.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_radio.xlsx
